In [32]:
import mlflow
import pandas as pd
import mlflow.sklearn
import logging
import time
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split   
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
import re
import os
from dotenv import load_dotenv
load_dotenv()
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [33]:
df=pd.read_csv('data.csv')
df.head(5)

,review,sentiment
0,Every great gangster movie has under-currents ...,positive
1,"I just saw this film last night, and I have to...",positive
2,This film is mildly entertaining if one neglec...,negative
3,Quentin Tarantino's partner in crime Roger Ava...,negative
4,I sat through this on TV hoping because of the...,negative


In [34]:
# Initialize once (IMPORTANT: avoids re-creating object for every row → faster)
lemmatizer = WordNetLemmatizer()

# Load default English stopwords (common words like "the", "is", etc.)
base_stopwords = set(stopwords.words('english'))

# Words we DO NOT want to remove because they affect sentiment meaning
# - Negations → flip sentiment (not good vs good)
# - Contrast words → change sentence meaning ("but")
# - Intensifiers → affect strength ("very good")
important_words = {
    'no', 'not', 'nor', 'never', 'none', 'nothing', 'nowhere', 'neither',
    'but', 'however', 'although', 'though', 'yet',
    'very', 'too', 'so', 'really', 'quite'
}

# Final stopwords = remove all default stopwords EXCEPT important ones
custom_stopwords = base_stopwords - important_words


# Function to expand contractions (critical for preserving negation meaning)
# Example:
# "don't like" → "do not like"
# Without this, "don't" → "dont" → negation lost
def expand_contractions(text):
    text = re.sub(r"n't", " not", text)   # don't → do not
    text = re.sub(r"'re", " are", text)   # you're → you are
    text = re.sub(r"'s", " is", text)     # it's → it is
    text = re.sub(r"'d", " would", text)  # I'd → I would
    text = re.sub(r"'ll", " will", text)  # I'll → I will
    text = re.sub(r"'t", " not", text)    # fallback for cases missed above
    text = re.sub(r"'ve", " have", text)  # I've → I have
    text = re.sub(r"'m", " am", text)     # I'm → I am
    return text


# Main preprocessing function
def normalize_text(text):
    
    # Remove leading/trailing spaces (safety cleanup)
    text = text.strip()
    
    # Convert everything to lowercase (standardization)
    text = text.lower()
    
    # Expand contractions BEFORE removing punctuation
    # (otherwise "don't" → "dont" → negation lost)
    text = expand_contractions(text)
    
    # Remove URLs (they don't help sentiment)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove punctuation (.,!?: etc.)
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numbers (usually not useful for sentiment)
    text = re.sub(r'\d+', '', text)
    
    # Split sentence into words (tokenization)
    words = text.split()
    
    # Remove stopwords + apply lemmatization
    # Lemmatization → converts words to base form (running → run)
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in custom_stopwords
    ]
    
    # Join words back into a single cleaned sentence
    return ' '.join(words)


# Apply preprocessing to the dataset
# Overwrites original column (no new column created)
df['review'] = df['review'].apply(normalize_text)

In [35]:
df.head(5)

,review,sentiment
0,every great gangster movie undercurrent human ...,positive
1,saw film last night say loved every minute tak...,positive
2,film mildly entertaining one neglect acknowled...,negative
3,quentin tarantino partner crime roger avary co...,negative
4,sat tv hoping name would worth timebut dear gu...,negative


In [36]:
df["sentiment"].value_counts()


sentiment
negative    269
positive    231
Name: count, dtype: int64

In [37]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [38]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
0,every great gangster movie undercurrent human ...,1
1,saw film last night say loved every minute tak...,1
2,film mildly entertaining one neglect acknowled...,0
3,quentin tarantino partner crime roger avary co...,0
4,sat tv hoping name would worth timebut dear gu...,0


In [39]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])  
y = df['sentiment']

In [46]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 11017 stored elements and shape (500, 100)>
  Coords	Values
  (0, 23)	1
  (0, 35)	1
  (0, 51)	1
  (0, 57)	4
  (0, 80)	1
  (0, 12)	4
  (0, 27)	5
  (0, 15)	3
  (0, 40)	3
  (0, 58)	1
  (0, 69)	2
  (0, 24)	1
  (0, 13)	4
  (0, 88)	3
  (0, 34)	2
  (0, 85)	1
  (0, 4)	1
  (0, 50)	1
  (0, 92)	1
  (0, 60)	4
  (0, 48)	1
  (0, 93)	1
  (0, 63)	2
  (0, 78)	1
  (0, 52)	1
  :	:
  (498, 38)	1
  (498, 46)	1
  (498, 30)	2
  (498, 17)	2
  (498, 9)	1
  (498, 11)	1
  (499, 57)	2
  (499, 80)	1
  (499, 27)	9
  (499, 40)	1
  (499, 13)	2
  (499, 34)	1
  (499, 92)	1
  (499, 78)	1
  (499, 52)	2
  (499, 82)	1
  (499, 98)	1
  (499, 70)	1
  (499, 10)	1
  (499, 79)	1
  (499, 1)	1
  (499, 87)	1
  (499, 53)	1
  (499, 11)	2
  (499, 25)	1


In [47]:
y

0      1
1      1
2      0
3      0
4      0
      ..
495    0
496    1
497    0
498    0
499    1
Name: sentiment, Length: 500, dtype: int64

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


In [41]:
import dagshub

tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
mlflow.set_tracking_uri(tracking_uri)
dagshub.init(repo_owner='georgeragan', repo_name='Sentiment-Analysis', mlflow=True)

2026-05-01 11:21:15,241 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/georgeragan/Sentiment-Analysis "HTTP/1.1 200 OK"


Initialized MLflow to track repo "georgeragan/Sentiment-Analysis"

2026-05-01 11:21:15,278 - INFO - Initialized MLflow to track repo "georgeragan/Sentiment-Analysis"


Repository georgeragan/Sentiment-Analysis initialized!

2026-05-01 11:21:15,286 - INFO - Repository georgeragan/Sentiment-Analysis initialized!


In [42]:
#import mlflow
#with mlflow.start_run():
  #mlflow.log_param('parameter name', 'value')
  #mlflow.log_metric('metric name', 1)

In [43]:
mlflow.set_experiment("Sentiment Analysis Experiment")

<Experiment: artifact_location='mlflow-artifacts:/585d51b25af9489bbda8269eecaab3c5', creation_time=1777612950853, experiment_id='1', last_update_time=1777612950853, lifecycle_stage='active', name='Sentiment Analysis Experiment', tags={}, trace_location=None, workspace='default'>

In [44]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("Experiment setup complete. Ready to train model.")

with mlflow.start_run():
    start_time=time.time()
    try:

        logging.info("Logging parameters and training model...")
        mlflow.log_param('vectorizer',"Bag of Words with max_features=50")
        mlflow.log_param('num_features', 100)
        mlflow.log_param('test_size', 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression()

        logging.info("Fitting model to training data...")
        model.fit(X_train, y_train)
        logging.info("Model training complete. Evaluating on test set...")
        mlflow.log_param('model_type', 'Logistic Regression')

        logging.info("Predicting test set labels...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        mlflow.log_metric('accuracy', acc)
        mlflow.log_metric('precision', precision)
        mlflow.log_metric('recall', recall)
        mlflow.log_metric('f1_score', f1)
        logging.info(f"Metrics logged: Accuracy={acc:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1 Score={f1:.4f}")

        logging.info("Saving model to MLflow...")
        mlflow.sklearn.log_model(model, "sentiment_model")  
        logging.info("Model saved successfully.")

        end_time = time.time()
        duration = end_time - start_time    
        mlflow.log_metric('training_duration_seconds', duration)
        logging.info(f"Training duration: {duration:.2f} seconds")
    
    except Exception as e:  
        logging.error(f"An error occurred during the experiment: {e}")



2026-05-01 11:21:16,362 - INFO - Experiment setup complete. Ready to train model.
2026-05-01 11:21:16,915 - INFO - Logging parameters and training model...
2026-05-01 11:21:17,913 - INFO - Initializing Logistic Regression model...
2026-05-01 11:21:17,919 - INFO - Fitting model to training data...
2026-05-01 11:21:18,008 - INFO - Model training complete. Evaluating on test set...
2026-05-01 11:21:18,320 - INFO - Predicting test set labels...
2026-05-01 11:21:18,320 - INFO - Calculating evaluation metrics...
2026-05-01 11:21:19,885 - INFO - Metrics logged: Accuracy=0.6640, Precision=0.6897, Recall=0.6250, F1 Score=0.6557
2026-05-01 11:21:19,891 - INFO - Saving model to MLflow...
2026/05/01 11:21:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 11:21:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, w

🏃 View run upset-pug-474 at: https://dagshub.com/georgeragan/Sentiment-Analysis.mlflow/#/experiments/1/runs/69ae35ed6c6e41d59d7b2a2c97cc1c08
🧪 View experiment at: https://dagshub.com/georgeragan/Sentiment-Analysis.mlflow/#/experiments/1
